# MODEL EVALUATION

In [1]:
import torch
import numpy as np

from PIL import Image
import torchvision.transforms.functional as TF

import utils
from model.hidden import Hidden
from noise_layers.noiser import Noiser

In [2]:
from noise_layers.cropout import Cropout
from noise_layers.crop import Crop
from noise_layers.dropout import Dropout
from noise_layers.resize import Resize
from noise_layers.identity import Identity

In [3]:
def load_image_for_hidden(image_path, device):
    """
    Load an image and return:
        original_tensor: original-resolution image in [-1, 1], [1, 3, H, W]
        hidden_tensor:   resized image in [-1, 1], [1, 3, 128, 128]
    """

    image_pil = Image.open(image_path).convert("RGB")

    # Original image
    original_tensor = TF.to_tensor(image_pil).to(device)
    original_tensor = original_tensor * 2 - 1
    original_tensor = original_tensor.unsqueeze(0)

    # Downsampled image for HiDDeN
    hidden_tensor = F.interpolate(
        original_tensor,
        size=(128, 128),
        mode="bilinear",
        align_corners=False
    )

    return original_tensor, hidden_tensor

In [4]:
OPTIONS_FILE = "./runs/test_6 2026.08.17--10-36-24/options-and-config.pickle"

CHECKPOINT_FILE = (
    "./runs/test_6 2026.08.17--10-36-24/"
    "checkpoints/test_6--epoch-300.pyt"
)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

train_options, hidden_config, noise_config = utils.load_options(
    OPTIONS_FILE
)

print(f"Model resolution: {hidden_config.H} x {hidden_config.W}")
print(f"Message length: {hidden_config.message_length}")

noiser = Noiser(noise_config, device)

hidden_net = Hidden(
    hidden_config,
    device,
    noiser,
    None
)

checkpoint = torch.load(
    CHECKPOINT_FILE,
    map_location=device
)

utils.model_from_checkpoint(
    hidden_net,
    checkpoint
)

print("Model loaded.")

Device: cuda
Model resolution: 128 x 128
Message length: 30
Model loaded.


In [2]:
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
import io


def identity(x):
    return x


def crop(x, scale=(0.7, 0.9)):
    """
    Random crop, then resize back to original dimensions.
    """
    _, _, h, w = x.shape

    min_scale, max_scale = scale
    crop_h = int(h * torch.empty(1).uniform_(min_scale, max_scale).item())
    crop_w = int(w * torch.empty(1).uniform_(min_scale, max_scale).item())

    top = torch.randint(0, h - crop_h + 1, (1,)).item()
    left = torch.randint(0, w - crop_w + 1, (1,)).item()

    cropped = x[:, :, top:top + crop_h, left:left + crop_w]

    return F.interpolate(
        cropped,
        size=(h, w),
        mode="bilinear",
        align_corners=False
    )


def cropout(image, cover, scale=(0.2, 0.2)):
    _, _, h, w = image.shape

    min_scale, max_scale = scale

    crop_h = int(h * torch.empty(1).uniform_(min_scale, max_scale).item())
    crop_w = int(w * torch.empty(1).uniform_(min_scale, max_scale).item())

    top = torch.randint(0, h - crop_h + 1, (1,)).item()
    left = torch.randint(0, w - crop_w + 1, (1,)).item()

    result = cover.clone()

    result[:, :, top:top + crop_h, left:left + crop_w] = \
        image[:, :, top:top + crop_h, left:left + crop_w]

    return result


def dropout(image, cover, keep_ratio=(0.2, 0.2)):
    keep = torch.empty(1).uniform_(
        keep_ratio[0], keep_ratio[1]
    ).item()

    mask = (torch.rand_like(image) < keep).float()

    return image * mask + cover * (1 - mask)


def resize(x, scale):
    """
    Downscale and then upscale back to original resolution.
    """
    _, _, h, w = x.shape

    min_scale, max_scale = scale
    s = torch.empty(1).uniform_(min_scale, max_scale).item()

    new_h = max(1, int(h * s))
    new_w = max(1, int(w * s))

    small = F.interpolate(
        x,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )

    return F.interpolate(
        small,
        size=(h, w),
        mode="bilinear",
        align_corners=False
    )

import io
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image


def jpeg(x, scale):
    """
    In-memory PIL JPEG compression attack for 128x128 images.
    
    Args:
        x (torch.Tensor): [B, 3, 128, 128] in range [-1, 1] or [0, 1].
        quality (int, float, or tuple): Quality factor between 10 and 100.
    """
    quality, _ = scale
    quality = quality * 100
    
    if isinstance(quality, (tuple, list)):
        q = int(torch.empty(1).uniform_(quality[0], quality[1]).item())
    else:
        q = int(quality)
    q = max(10, min(100, q))

    is_neg1_to_1 = x.min() < -0.1

    if is_neg1_to_1:
        x_255 = ((x.clamp(-1.0, 1.0) + 1.0) * 127.5).round()
    else:
        x_255 = (x.clamp(0.0, 1.0) * 255.0).round()

    out = torch.empty_like(x)

    for i in range(x.shape[0]):
        np_img = x_255[i].byte().permute(1, 2, 0).cpu().numpy()
        pil_img = Image.fromarray(np_img)

        buffer = io.BytesIO()
        pil_img.save(buffer, format="JPEG", quality=q)
        buffer.seek(0)
        decompressed_img = Image.open(buffer)

        t = TF.to_tensor(decompressed_img).to(x.device)
        out[i] = t * 2.0 - 1.0 if is_neg1_to_1 else t

    return out


def jpeg_differentiable(x, quality=(10, 100)):
    """
    Fast, differentiable DCT-based JPEG attack tailored for 128x128 tensors in [-1, 1].
    """
    if isinstance(quality, (tuple, list)):
        q = float(torch.empty(1).uniform_(quality[0], quality[1]).item())
    else:
        q = float(quality)
    q = max(10.0, min(100.0, q))

    scale_factor = 2.0 - q * 0.02 if q >= 50.0 else 50.0 / q
    device = x.device

    # Standard 8x8 DCT matrix
    coff = torch.zeros((8, 8), dtype=torch.float32, device=device)
    coff[0, :] = 1.0 * np.sqrt(1.0 / 8.0)
    for i in range(1, 8):
        for j in range(8):
            coff[i, j] = np.cos(np.pi * i * (2 * j + 1) / 16.0) * np.sqrt(2.0 / 8.0)

    # Standard Luminance Quantization Table
    lum_table = (torch.tensor([
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99]
    ], dtype=torch.float32, device=device) * scale_factor).round().clamp(min=1.0)

    # [-1, 1] -> [0, 255]
    img_255 = (x.clamp(-1.0, 1.0) + 1.0) * 127.5

    # RGB to YUV
    y = 0.299 * img_255[:, 0:1] + 0.587 * img_255[:, 1:2] + 0.114 * img_255[:, 2:3]
    u = -0.1687 * img_255[:, 0:1] - 0.3313 * img_255[:, 1:2] + 0.5 * img_255[:, 2:3]
    v = 0.5 * img_255[:, 0:1] - 0.4187 * img_255[:, 1:2] - 0.0813 * img_255[:, 2:3]
    yuv = torch.cat([y, u, v], dim=1)

    # 8x8 DCT on 128x128 (16x16 blocks)
    blocks = torch.cat(torch.cat(yuv.split(8, dim=2), dim=0).split(8, dim=3), dim=0)
    dct_blocks = torch.matmul(torch.matmul(coff, blocks), coff.t())

    # Quantize & Dequantize
    q_dct = torch.round(dct_blocks / lum_table) * lum_table

    # IDCT
    idct_blocks = torch.matmul(torch.matmul(coff.t(), q_dct), coff)
    yuv_rec = torch.cat(torch.cat(idct_blocks.chunk(256, dim=0), dim=3).chunk(16, dim=0), dim=2)

    # YUV to RGB
    r = yuv_rec[:, 0:1] + 1.40198758 * yuv_rec[:, 2:3]
    g = yuv_rec[:, 0:1] - 0.344113281 * yuv_rec[:, 1:2] - 0.714103821 * yuv_rec[:, 2:3]
    b = yuv_rec[:, 0:1] + 1.77197812 * yuv_rec[:, 1:2]
    rgb_rec = torch.cat([r, g, b], dim=1)

    return (rgb_rec / 127.5 - 1.0).clamp(-1.0, 1.0)

In [7]:
# import torch
# import torch.nn.functional as F
# import torchvision.transforms.functional as TF
# import random

# def adjust_sharpness(x: torch.Tensor, factor=(1.5, 3.0)) -> torch.Tensor:
#     """Enhances image sharpness (unsharp masking / high-pass boost)."""
#     f = random.uniform(factor[0], factor[1])
#     return TF.adjust_sharpness(x, sharpness_factor=f)

# def adjust_contrast(x: torch.Tensor, factor=(1.2, 1.8)) -> torch.Tensor:
#     """Enhances image contrast."""
#     f = random.uniform(factor[0], factor[1])
#     return TF.adjust_contrast(x, contrast_factor=f)

# def adjust_saturation(x: torch.Tensor, factor=(1.2, 2.0)) -> torch.Tensor:
#     """Boosts or alters color saturation (vivid filter effect)."""
#     f = random.uniform(factor[0], factor[1])
#     return TF.adjust_saturation(x, saturation_factor=f)

# def color_temperature(x: torch.Tensor, mode='random') -> torch.Tensor:
#     """
#     Simulates warm (golden hour) or cool (cinematic blue) tone grading.
#     x is assumed to be in range [0, 1].
#     """
#     shift_mode = mode if mode != 'random' else random.choice(['warm', 'cool'])
#     out = x.clone()
#     if shift_mode == 'warm':
#         # Boost Red/Yellow, slightly attenuate Blue
#         r_boost = random.uniform(1.05, 1.20)
#         b_drop = random.uniform(0.85, 0.95)
#         out[:, 0, :, :] = out[:, 0, :, :] * r_boost
#         out[:, 2, :, :] = out[:, 2, :, :] * b_drop
#     else:
#         # Boost Blue/Cyan, slightly attenuate Red
#         b_boost = random.uniform(1.05, 1.20)
#         r_drop = random.uniform(0.85, 0.95)
#         out[:, 2, :, :] = out[:, 2, :, :] * b_boost
#         out[:, 0, :, :] = out[:, 0, :, :] * r_drop
#     return torch.clamp(out, 0.0, 1.0)

# def vintage_fade(x: torch.Tensor, lift=(0.05, 0.15), contrast=(0.85, 0.95)) -> torch.Tensor:
#     """Simulates matte/faded film preset (lifted black levels, muted contrast)."""
#     lift_val = random.uniform(lift[0], lift[1])
#     c_factor = random.uniform(contrast[0], contrast[1])
#     out = TF.adjust_contrast(x, contrast_factor=c_factor)
#     out = out * (1.0 - lift_val) + lift_val
#     return torch.clamp(out, 0.0, 1.0)

# def vignette_filter(x: torch.Tensor, strength_range=(0.3, 0.7)) -> torch.Tensor:
#     """Applies a radial darkening vignette towards image corners."""
#     strength = random.uniform(strength_range[0], strength_range[1])
#     *_, H, W = x.shape
#     y = torch.linspace(-1, 1, H, device=x.device).view(H, 1).repeat(1, W)
#     x_coord = torch.linspace(-1, 1, W, device=x.device).view(1, W).repeat(H, 1)
#     radius = torch.sqrt(x_coord**2 + y**2) / 1.414  # Normalize [0, 1]
    
#     mask = 1.0 - strength * torch.clamp(radius, 0.0, 1.0) ** 2
#     mask = mask.unsqueeze(0).unsqueeze(0) if x.dim() == 4 else mask.unsqueeze(0)
#     return torch.clamp(x * mask, 0.0, 1.0)

# def social_media_beautify_preset(x: torch.Tensor) -> torch.Tensor:
#     """Combines moderate contrast, sharpness, and saturation as an editing pipeline."""
#     out = adjust_sharpness(x, factor=(1.5, 2.2))
#     out = adjust_contrast(out, factor=(1.1, 1.3))
#     out = adjust_saturation(out, factor=(1.15, 1.4))
#     return out

In [8]:
# attacks = {
#     # Geometric & Spatial Degradations
#     "Identity": identity,
#     "Crop": lambda x: crop(x, scale=(0.7, 0.7)),
#     "Cropout": lambda image, cover: cropout(image, cover, scale=(0.2, 0.2)),
#     "Dropout": lambda image, cover: dropout(image, cover, keep_ratio=(0.2, 0.2)),
#     "Resize": lambda x: resize(x, scale=(0.5, 0.9)),

#     # # Aesthetic & Editing Filters
#     # "Contrast_Enhance": lambda x: adjust_contrast(x, factor=(1.2, 1.8)),
#     # "Sharpness_Enhance": lambda x: adjust_sharpness(x, factor=(1.8, 3.0)),
#     # "Vivid_Saturation": lambda x: adjust_saturation(x, factor=(1.3, 2.0)),
#     # "Color_Grading_Warm": lambda x: color_temperature(x, mode='warm'),
#     # "Color_Grading_Cool": lambda x: color_temperature(x, mode='cool'),
#     # "Vintage_Fade": lambda x: vintage_fade(x, lift=(0.08, 0.18)),
#     # "Vignette": lambda x: vignette_filter(x, strength_range=(0.4, 0.8)),
#     # "Full_Beautify_Pipeline": lambda x: social_media_beautify_preset(x),
# }

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import kornia
from kornia.losses import ssim_loss

def run_inference(
    image_path,
    message,
    hidden_net,
    hidden_config,
    device,
    attacks,
    attack_range
):
    # ============================================================
    # Load image
    # ============================================================
    original_image, image_tensor = load_image_for_hidden(
        image_path,
        device
    )

    # ============================================================
    # Validate message
    # ============================================================
    if len(message) != hidden_config.message_length:
        raise ValueError(
            f"Message length is {len(message)}, "
            f"but model expects {hidden_config.message_length} bits."
        )

    if not set(message).issubset({"0", "1"}):
        raise ValueError("Message must contain only 0 and 1.")

    message_tensor = torch.tensor(
        [[float(bit) for bit in message]],
        dtype=torch.float32,
        device=device
    )

    hidden_net.encoder_decoder.eval()
    hidden_net.discriminator.eval()
    atk = attacks[0] if attacks is not None else None

    with torch.no_grad():

        # ========================================================
        # Encode
        # ========================================================
        encoded_images = hidden_net.encoder_decoder.encode(
            image_tensor,
            message_tensor
        )

        # ========================================================
        # Image quality BEFORE attacks
        # Match MBRS evaluation
        # ========================================================
        psnr_loss = kornia.losses.psnr_loss(
            encoded_images.detach(),
            image_tensor,
            2
        ).item()

        psnr = -psnr_loss

        ssim = 1 - 2 * ssim_loss(
            encoded_images.detach(),
            image_tensor,
            window_size=5,
            reduction="mean"
        )

        # ========================================================
        # Apply attacks
        # ========================================================
        attacked_image = encoded_images

        if attacks is not None:

            for attack in attacks:
                if attack == "Crop":
                    attacked_image = crop(
                        attacked_image,
                        scale=attack_range
                    )

                elif attack == "Cropout":
                    attacked_image = cropout(
                        attacked_image,
                        image_tensor,
                        scale=attack_range
                    )

                elif attack == "Dropout":
                    attacked_image = dropout(
                        attacked_image,
                        image_tensor,
                        keep_ratio=attack_range
                    )

                elif attack == "Resize":
                    attacked_image = resize(
                        attacked_image,
                        attack_range
                    )

                elif attack == "JPEG":
                    attacked_image = jpeg(
                        attacked_image,
                        attack_range
                    )

                else:
                    raise ValueError(
                        f"Unknown attack: {attack}"
                    )

        # ========================================================
        # Decode attacked image
        # ========================================================
        decoded_messages = hidden_net.encoder_decoder.decode(
            attacked_image
        )

    # ============================================================
    # BER
    # ============================================================
    decoded_rounded = (
        decoded_messages
        .detach()
        .cpu()
        .numpy()
        .round()
        .clip(0, 1)
    )

    original_message = (
        message_tensor
        .detach()
        .cpu()
        .numpy()
    )

    bit_errors = int(
        np.sum(
            np.abs(
                decoded_rounded - original_message
            )
        )
    )

    batch_size = image_tensor.shape[0]

    total_bits = (
        batch_size *
        message_tensor.shape[1]
    )

    ber = bit_errors / total_bits
    # ber_float = ber.detach().cpu().item()

    # print(
    #     f"BER={ber_float}, "
    #     f"PSNR={psnr}, "
    #     f"SSIM={ssim}"
    # )

    # if ber > 0:
    #     print("message :", message_tensor.cpu().numpy())
    #     print("decoded :", decoded_messages.cpu().numpy())
    #     print("rounded :", decoded_messages.round().cpu().numpy())

    return [atk, ber, psnr, ssim]

In [1]:
import torch
import torch.nn.functional as F
import numpy as np


def run_inference_residual(
    image_path,
    message,
    hidden_net,
    hidden_config,
    device,
    attacks,
):
    # ============================================================
    # Load image
    # ============================================================
    original_image, image_tensor = load_image_for_hidden(
        image_path,
        device
    )

    # ============================================================
    # Validate message
    # ============================================================
    if len(message) != hidden_config.message_length:
        raise ValueError(
            f"Message length is {len(message)}, "
            f"but model expects {hidden_config.message_length} bits."
        )

    if not set(message).issubset({"0", "1"}):
        raise ValueError("Message must contain only 0 and 1.")

    message_tensor = torch.tensor(
        [[float(bit) for bit in message]],
        dtype=torch.float32,
        device=device
    )

    hidden_net.encoder_decoder.eval()
    hidden_net.discriminator.eval()

    # ============================================================
    # Encode
    # ============================================================
    with torch.no_grad():

        encoded_images = hidden_net.encoder_decoder.encode(
            image_tensor,
            message_tensor
        )

        decoded_messages = hidden_net.encoder_decoder.decode(
            encoded_images
        )

        # --------------------------------------------------------
        # Encoder reconstruction loss
        # --------------------------------------------------------
        if hidden_net.vgg_loss is None:
            g_loss_enc = hidden_net.mse_loss(
                encoded_images,
                image_tensor
            )
        else:
            vgg_on_cov = hidden_net.vgg_loss(image_tensor)
            vgg_on_enc = hidden_net.vgg_loss(encoded_images)

            g_loss_enc = hidden_net.mse_loss(
                vgg_on_cov,
                vgg_on_enc
            )

        # --------------------------------------------------------
        # Decoder loss
        # --------------------------------------------------------
        g_loss_dec = hidden_net.mse_loss(
            decoded_messages,
            message_tensor
        )

        # --------------------------------------------------------
        # Discriminator metrics
        # --------------------------------------------------------
        batch_size = image_tensor.shape[0]

        d_target_label_cover = torch.full(
            (batch_size, 1),
            hidden_net.cover_label,
            device=device,
            dtype=torch.float32
        )

        d_target_label_encoded = torch.full(
            (batch_size, 1),
            hidden_net.encoded_label,
            device=device,
            dtype=torch.float32
        )

        g_target_label_encoded = torch.full(
            (batch_size, 1),
            hidden_net.cover_label,
            device=device,
            dtype=torch.float32
        )

        # Clean image → discriminator
        d_on_cover = hidden_net.discriminator(
            image_tensor
        )

        d_loss_on_cover = hidden_net.bce_with_logits_loss(
            d_on_cover,
            d_target_label_cover
        )

        # Encoded image → discriminator
        d_on_encoded = hidden_net.discriminator(
            encoded_images
        )

        d_loss_on_encoded = hidden_net.bce_with_logits_loss(
            d_on_encoded,
            d_target_label_encoded
        )

        # Generator adversarial loss
        d_on_encoded_for_enc = hidden_net.discriminator(
            encoded_images
        )

        g_loss_adv = hidden_net.bce_with_logits_loss(
            d_on_encoded_for_enc,
            g_target_label_encoded
        )

    # ============================================================
    # Encoder-level metrics
    # ============================================================

    decoded_rounded = (
        decoded_messages
        .detach()
        .cpu()
        .numpy()
        .round()
        .clip(0, 1)
    )

    original_message = (
        message_tensor
        .detach()
        .cpu()
        .numpy()
    )

    bit_errors = int(
        np.sum(
            np.abs(
                decoded_rounded -
                original_message
            )
        )
    )

    total_bits = (
        batch_size *
        message_tensor.shape[1]
    )

    ber = bit_errors / total_bits

    # ============================================================
    # Residual reconstruction
    # ============================================================

    with torch.no_grad():

        residual_128 = (
            encoded_images -
            image_tensor
        )

        import torchvision.utils as vutils

        # Normalize residual from [-max, +max] to [0, 1] (0.5 is neutral gray)
        res_vis = (residual_128[0] - residual_128[0].min()) / (residual_128[0].max() - residual_128[0].min() + 1e-8)

        # Or multiply residual by a fixed factor (e.g. 10x) centered at 0.5
        # res_vis = torch.clamp(residual_128[0] * 10.0 + 0.5, 0.0, 1.0)

        vutils.save_image(res_vis, "residual_amplified.png")
        print("Saved amplified residual visualization to residual_amplified.png")

        residual_hr = F.interpolate(
            residual_128,
            size=original_image.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        ALPHA = 1.0

        watermarked_hr = (
            original_image +
            ALPHA * residual_hr
        ).clamp(-1, 1)

    # ============================================================
    # Base metrics
    # ============================================================

    base_metrics = {
        "encoder_mse": g_loss_enc.item(),
        "dec_mse": g_loss_dec.item(),
        "adversarial_bce": g_loss_adv.item(),
        "discr_cover_bce": d_loss_on_cover.item(),
        "discr_encoded_bce": d_loss_on_encoded.item(),
    }

    # ============================================================
    # Attack loop
    # ============================================================

    results = {}

    with torch.no_grad():
        watermarked_128 = F.interpolate(
            watermarked_hr,
            size=(hidden_config.H, hidden_config.W),
            mode="bilinear",
            align_corners=False
        )

        pipeline_error = F.mse_loss(
            watermarked_128,
            encoded_images
        ).item()

        residual_error = F.mse_loss(
            watermarked_128 - image_tensor,
            encoded_images - image_tensor
        ).item()

    for attack_name, attack_fn in attacks.items():

        with torch.no_grad():


            # attacked_hr = watermarked_hr.clone()

            # Final HR watermark → attack
            attacked_hr = attack_fn(
                watermarked_hr.clone()
            )

            # Attack result → HiDDeN resolution
            attacked_128 = F.interpolate(
                attacked_hr,
                size=(
                    hidden_config.H,
                    hidden_config.W
                ),
                mode="bilinear",
                align_corners=False
            )

            direct_encoded = encoded_images

            reconstructed_128 = F.interpolate(
                watermarked_hr,
                size=(128, 128),
                mode="bilinear",
                align_corners=False
            )

            pipeline_error = torch.mean(
                (reconstructed_128 - direct_encoded) ** 2
            )

            print(f"Pipeline error (MSE) for {attack_name}: {pipeline_error.item()}")
    
            # Decode
            decoded_messages = (
                hidden_net.encoder_decoder.decode(
                    attacked_128
                )
            )

        decoded_rounded = (
            decoded_messages
            .detach()
            .cpu()
            .numpy()
            .round()
            .clip(0, 1)
        )

        bit_errors = int(
            np.sum(
                np.abs(
                    decoded_rounded -
                    original_message
                )
            )
        )

        ber = bit_errors / total_bits

        results[attack_name] = {
            **base_metrics,

            "ber": ber,
            "bit_errors": bit_errors,
            "total_bits": total_bits,

            "attack_mse": torch.mean(
                (attacked_128 - watermarked_128) ** 2
            ).item(),
            "residual_error": residual_error,
            "pipeline_error": pipeline_error,
        }

    return results

In [11]:
import importlib
import model.hidden

importlib.reload(model.hidden)

hidden_net = model.hidden.Hidden(
    hidden_config,
    device,
    None,
    None
)

checkpoint = torch.load(
    CHECKPOINT_FILE,
    map_location=device
)

utils.model_from_checkpoint(
    hidden_net,
    checkpoint
)

print(hasattr(hidden_net, "validate_on_batch_custom_attacks"))

True


# BULK

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np


def run_bulk_inference(
    image_dir,
    message,
    hidden_net,
    hidden_config,
    device,
    attack,
    attack_range
):
    """
    Run inference on every image in image_dir.

    Each image is tested against every attack.

    Returns:
        DataFrame with:
            image
            attack
            ber
            bit_errors
            total_bits
    """

    image_dir = Path(image_dir)

    image_extensions = {
        ".jpg", ".jpeg", ".png", ".webp", ".bmp"
    }

    image_paths = sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    )

    # print(f"Found {len(image_paths)} images")
    # print(f"Attacks: {len(attacks)}")
    # print(f"Total inference cases: {len(image_paths) * len(attacks)}")
    # print("-" * 60)

    rows = []
    print(len(image_paths))
    for i, image_path in enumerate(image_paths, 1):

        # print(
        #     f"[{i}/{len(image_paths)}] "
        #     f"{image_path.name}"
        # )

        results = run_inference(
            image_path,
            message,
            hidden_net,
            hidden_config,
            device,
            [attack],
            attack_range
        )


        rows.append({
            "image": image_path.name,
            "attack_eval": results[0],

            # Watermark recovery
            "ber": results[1],
            "psnr": results[2],
            "ssim": results[3].item(),
            # "bit_errors": result["bit_errors"],
            # "total_bits": result["total_bits"],

            # # HiDDeN metrics
            # "encoder_mse": result["encoder_mse"],
            # "dec_mse": result["dec_mse"],
            # "adversarial_bce": result["adversarial_bce"],
            # "discr_cover_bce": result["discr_cover_bce"],
            # "discr_encoded_bce": result["discr_encoded_bce"],

            # Attack
            # "attack_mse": result["attack_mse"],
            # "residual_error": result["residual_error"],
            # "pipeline_error": result["pipeline_error"],
        })

    df = pd.DataFrame(rows)

    return {
    "model": "HiDDeN",
    "attack": attack,
    "severity": attack_range,
    "retained_area": attack_range[0] ** 2
        if attack in ["Crop", "Cropout"]
        else attack_range[0],
    "BER": df["ber"].mean(),
    "PSNR": df["psnr"].mean(),
    "SSIM": df["ssim"].mean(),
}

In [ ]:
IMAGE_DIR = "./official_images"

MESSAGE = "010100110100000100101010000000"

results = {}

ranges = [
    (0.9, 0.9),
    (0.8, 0.8),
    (0.7, 0.7),
    (0.6, 0.6),
    (0.5, 0.5),
    (0.4, 0.4),
    (0.3, 0.3),
    (0.2, 0.2),
    (0.1, 0.1),
]

for attack_range in ranges:

    print("=" * 60)
    print(f"Running Dropout {attack_range}")
    print("=" * 60)

    results_df = run_bulk_inference(
        image_dir=IMAGE_DIR,
        message=MESSAGE,
        hidden_net=hidden_net,
        hidden_config=hidden_config,
        device=device,
        attack="JPEG",
        attack_range=attack_range,
    )

    results[attack_range] = results_df

Running Dropout (0.9, 0.9)
217
Running Dropout (0.8, 0.8)
217
Running Dropout (0.7, 0.7)
217
Running Dropout (0.6, 0.6)
217
Running Dropout (0.5, 0.5)
217
Running Dropout (0.4, 0.4)
217
Running Dropout (0.3, 0.3)
217
Running Dropout (0.2, 0.2)
217
Running Dropout (0.1, 0.1)
217


In [14]:
results

{(0.9, 0.9): {'model': 'HiDDeN',
  'attack': 'Resize',
  'severity': (0.9, 0.9),
  'retained_area': 0.9,
  'BER': np.float64(0.059139784946236555),
  'PSNR': np.float64(33.00171190253051),
  'SSIM': np.float64(0.8931910620856395)},
 (0.8, 0.8): {'model': 'HiDDeN',
  'attack': 'Resize',
  'severity': (0.8, 0.8),
  'retained_area': 0.8,
  'BER': np.float64(0.10522273425499233),
  'PSNR': np.float64(33.00171190253051),
  'SSIM': np.float64(0.8931910620856395)},
 (0.7, 0.7): {'model': 'HiDDeN',
  'attack': 'Resize',
  'severity': (0.7, 0.7),
  'retained_area': 0.7,
  'BER': np.float64(0.1857142857142857),
  'PSNR': np.float64(33.00171190253051),
  'SSIM': np.float64(0.8931910620856395)},
 (0.6, 0.6): {'model': 'HiDDeN',
  'attack': 'Resize',
  'severity': (0.6, 0.6),
  'retained_area': 0.6,
  'BER': np.float64(0.2726574500768049),
  'PSNR': np.float64(33.00171190253051),
  'SSIM': np.float64(0.8931910620856395)},
 (0.5, 0.5): {'model': 'HiDDeN',
  'attack': 'Resize',
  'severity': (0.5, 0.

In [15]:
# print("OVERALL")
# print("=" * 50)

# metrics = [
#     ("BER", "ber"),
#     ("PSNR", "psnr"),
#     ("SSIM", "ssim"),
#     # ("Bit Errors", "bit_errors"),
#     # ("Encoder MSE", "encoder_mse"),
#     # ("Decoder MSE", "dec_mse"),
#     # ("Adversarial BCE", "adversarial_bce"),
#     # ("Discriminator Cover BCE", "discr_cover_bce"),
#     # ("Discriminator Encoded BCE", "discr_encoded_bce"),
#     # ("Attack MSE", "attack_mse"),
#     # ("Residual MSE", "residual_error"),
#     # ("Pipeline MSE", "pipeline_error"),
# ]

# for display_name, column in metrics:
#     print(f"\n{display_name}:")
#     print(f"  Mean: {results_df[column].mean():.6f}")
#     print(f"  Min : {results_df[column].min():.6f}")
#     print(f"  Max : {results_df[column].max():.6f}")

In [16]:
# results_df.groupby("attack").agg(
#     mean_ber=("ber", "mean"),
#     median_ber=("ber", "median"),
#     max_ber=("ber", "max"),
#     mean_errors=("bit_errors", "mean"),
#     perfect=("ber", lambda x: (x == 0).mean()),
# )

In [17]:
import os
import time
import threading
import psutil
import numpy as np
import torch
from PIL import Image
import torchvision.transforms.functional as TF


def benchmark_inference(
    image_path,
    hidden_net,
    hidden_config,
    device,
    alpha=1.0,
    warmup=3,
    runs=20
):
    """
    Benchmark CPU resource usage and latency for the HR watermark pipeline.

    Measures:
        - Average inference time
        - Min / max inference time
        - Peak process RAM
        - Average CPU utilization
        - CPU count
    """

    # ---------------------------------------------------------
    # Force CPU
    # ---------------------------------------------------------
    if device.type != "cpu":
        print("WARNING: device is not CPU.")
        print("This benchmark is intended for CPU inference.")

    process = psutil.Process(os.getpid())

    # ---------------------------------------------------------
    # Load image
    # ---------------------------------------------------------
    image_pil = Image.open(image_path).convert("RGB")

    original_image = TF.to_tensor(image_pil).to(device)
    original_image = original_image * 2 - 1
    original_image = original_image.unsqueeze(0)

    hidden_image = TF.resize(
        original_image,
        [hidden_config.H, hidden_config.W],
        interpolation=TF.InterpolationMode.BILINEAR
    )

    # ---------------------------------------------------------
    # Message
    # ---------------------------------------------------------
    message = torch.tensor(
        [[float(bit) for bit in MESSAGE]],
        dtype=torch.float32,
        device=device
    )

    # ---------------------------------------------------------
    # Warmup
    # ---------------------------------------------------------
    hidden_net.encoder_decoder.to(device)
    hidden_net.discriminator.to(device)

    hidden_net.encoder_decoder.eval()
    hidden_net.discriminator.eval()

    with torch.no_grad():
        for _ in range(warmup):

            encoded = hidden_net.encoder_decoder.encode(
                hidden_image,
                message
            )

            residual = encoded - hidden_image

            residual_hr = torch.nn.functional.interpolate(
                residual,
                size=original_image.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            watermarked = (
                original_image +
                alpha * residual_hr
            ).clamp(-1, 1)

            watermarked_128 = torch.nn.functional.interpolate(
                watermarked,
                size=(hidden_config.H, hidden_config.W),
                mode="bilinear",
                align_corners=False
            )

            decoded = hidden_net.encoder_decoder.decode(
                watermarked_128
            )

    # ---------------------------------------------------------
    # CPU monitoring
    # ---------------------------------------------------------
    cpu_samples = []
    stop_monitor = False

    def monitor_cpu():
        while not stop_monitor:
            cpu_samples.append(
                process.cpu_percent(interval=0.1)
            )

    monitor_thread = threading.Thread(
        target=monitor_cpu,
        daemon=True
    )

    monitor_thread.start()

    # ---------------------------------------------------------
    # Benchmark
    # ---------------------------------------------------------
    times = []

    peak_ram = process.memory_info().rss

    with torch.no_grad():

        for _ in range(runs):

            start = time.perf_counter()

            # 1. Encode at 128x128
            encoded = hidden_net.encoder_decoder.encode(
                hidden_image,
                message
            )

            # 2. Extract residual
            residual = encoded - hidden_image

            # 3. Upscale residual to original resolution
            residual_hr = torch.nn.functional.interpolate(
                residual,
                size=original_image.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            # 4. Overlay on original
            watermarked = (
                original_image +
                alpha * residual_hr
            ).clamp(-1, 1)

            # 5. Resize back to 128x128
            watermarked_128 = torch.nn.functional.interpolate(
                watermarked,
                size=(hidden_config.H, hidden_config.W),
                mode="bilinear",
                align_corners=False
            )

            # 6. Decode
            decoded = hidden_net.encoder_decoder.decode(
                watermarked_128
            )

            elapsed = time.perf_counter() - start

            times.append(elapsed)

            ram = process.memory_info().rss
            peak_ram = max(peak_ram, ram)

    stop_monitor = True
    monitor_thread.join(timeout=1)

    # ---------------------------------------------------------
    # Results
    # ---------------------------------------------------------
    ram_mb = peak_ram / (1024 ** 2)

    avg_time = np.mean(times)
    min_time = np.min(times)
    max_time = np.max(times)

    avg_cpu = (
        np.mean(cpu_samples)
        if cpu_samples
        else 0
    )

    print("=" * 60)
    print("CPU INFERENCE BENCHMARK")
    print("=" * 60)

    print(f"Image              : {image_pil.size[0]} × {image_pil.size[1]}")
    print(f"Runs               : {runs}")
    print(f"CPU cores          : {os.cpu_count()}")

    print()
    print(f"Average latency    : {avg_time * 1000:.2f} ms")
    print(f"Min latency        : {min_time * 1000:.2f} ms")
    print(f"Max latency        : {max_time * 1000:.2f} ms")
    print(f"Throughput         : {1 / avg_time:.2f} images/sec")

    print()
    print(f"Average CPU usage  : {avg_cpu:.1f}%")
    print(f"Peak process RAM   : {ram_mb:.1f} MB")

    print("=" * 60)

    return {
        "resolution": image_pil.size,
        "avg_latency_ms": avg_time * 1000,
        "min_latency_ms": min_time * 1000,
        "max_latency_ms": max_time * 1000,
        "throughput_img_s": 1 / avg_time,
        "avg_cpu_percent": avg_cpu,
        "peak_ram_mb": ram_mb,
        "cpu_cores": os.cpu_count(),
    }

In [18]:
MESSAGE = "010100110100000100101010000000"

import torch
device = torch.device("cuda")

results_1k = benchmark_inference(
    image_path="./official_images/007.jpg",
    hidden_net=hidden_net,
    hidden_config=hidden_config,
    device=device,
    alpha=1.0
)

This benchmark is intended for CPU inference.
CPU INFERENCE BENCHMARK
Image              : 1365 × 768
Runs               : 20
CPU cores          : 20

Average latency    : 1.09 ms
Min latency        : 1.07 ms
Max latency        : 1.21 ms
Throughput         : 919.29 images/sec

Average CPU usage  : 19.9%
Peak process RAM   : 1381.3 MB


In [19]:
import datetime
import re

file_path = "/home/trinh.minh.hieu/HiDDeN/runs/test_6 2026.08.17--10-36-24/test_6.log"

# Matches patterns like "Epoch 5 training duration 141.32 sec" or "...duration: 141 sec"
pattern = re.compile(r"Epoch\s+\d+.*?(?:duration|:)\s*([0-9.]+)\s*s(?:ec)?", re.IGNORECASE)

total_seconds = 0.0
count = 0

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        match = pattern.search(line)
        if match:
            total_seconds += float(match.group(1))
            count += 1

formatted_time = str(datetime.timedelta(seconds=round(total_seconds)))

print(f"Extracted epochs : {count}")
print(f"Total time (s)   : {total_seconds:.2f} seconds")
print(f"Formatted time   : {formatted_time} (HH:MM:SS)")

Extracted epochs : 300
Total time (s)   : 42489.55 seconds
Formatted time   : 11:48:10 (HH:MM:SS)
